# 🚑 Emergency Medical Supply Distribution Optimization

## Project Goal

This project aims to optimize the delivery of emergency medical supplies from a central hospital to multiple clinics.

The workflow consists of:

1. Predicting travel time using a trained XGBoost model
2. Using a Genetic Algorithm to find the optimal delivery sequence
3. Minimizing total travel time

This helps emergency response teams distribute medical supplies more efficiently during disasters and public health emergencies.

In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import datetime
import random
import pickle
from copy import copy

In [2]:
# Load trained XGB model
loaded_model = pickle.load(
    open("C:/Projects/14_Emergency_RL_System/model/model.pkl","rb")
)

print("Model loaded successfully")

Model loaded successfully


In [3]:
# Define Emergency Supply Locations
supply_locations = {
    "Central_Hospital": (40.7580, -73.9855),

    "Clinic_A": (40.7484, -73.9857),
    "Clinic_B": (40.7306, -73.9352),
    "Clinic_C": (40.7128, -74.0060),
    "Clinic_D": (40.7060, -74.0086),
    "Clinic_E": (40.7614, -73.9776),

    "Clinic_F": (40.7789, -73.9682),
    "Clinic_G": (40.7527, -73.9772),
    "Clinic_H": (40.7420, -74.0018),
    "Clinic_I": (40.7218, -73.9980),
    "Clinic_J": (40.7003, -73.9419),
    "Clinic_K": (40.6892, -74.0445)
}

In [4]:
# Define Delivery Date
date_list = [4, 6, 2016]

year = date_list[2]
month = date_list[1]
day = date_list[0]

delivery_date = datetime.date(
    year,
    month,
    day
)

In [5]:
# Predict travel time between two locations
def travel_time_between_locations(
    point1,
    point2,
    hour=10,
    passenger_count=1,
    store_and_fwd_flag=0,
    pickup_minute=0
):
    # Simple Euclidean distance approximation
    trip_distance = np.linalg.norm(
    np.array(point2) - np.array(point1)
)
        
    model_data = {'passenger_count': passenger_count,
                  'pickup_longitude' : point1[1],
                  'pickup_latitude' : point1[0],
                  'dropoff_longitude' : point2[1],
                  'dropoff_latitude' : point2[0],
                  'store_and_fwd_flag' : store_and_fwd_flag,
                  'pickup_month' : delivery_date.month,
                  'pickup_day' : delivery_date.day,
                  'pickup_weekday' : delivery_date.weekday(),
                  'pickup_hour': hour,
                  'pickup_minute' : pickup_minute,
                  'latitude_difference' : point2[0] - point1[0],
                  'longitude_difference' : point2[1] - point1[1],
                  'trip_distance' : trip_distance,
                  'is_rush_hour': 1 if hour in [7,8,9,16,17,18] else 0,
                  'is_weekend': 1 if delivery_date.weekday() >= 5 else 0
                 }

    df = pd.DataFrame([model_data])

    pred = loaded_model.predict(df)
    
    return pred[0]

In [6]:
# Generate Random Delivery Routes
def create_guess(points):

    guess = copy(points)

    np.random.shuffle(guess)

    guess.append(guess[0])

    return list(guess)

In [7]:
create_guess(
    list(supply_locations.keys())
)

['Clinic_F',
 'Clinic_K',
 'Clinic_C',
 'Clinic_D',
 'Clinic_E',
 'Clinic_A',
 'Clinic_B',
 'Clinic_H',
 'Clinic_G',
 'Clinic_I',
 'Central_Hospital',
 'Clinic_J',
 'Clinic_F']

In [8]:
# Create Initial Population
def create_generation(
    points,
    population=100
):

    generation = [
        create_guess(points)
        for _ in range(population)
    ]

    return generation

In [9]:
# Fitness Function ( Lower travel time = Better Route)
coordinates = supply_locations

In [10]:
def fitness_score(route):

    score = 0

    for ix, point_id in enumerate(route[:-1]):

        score += travel_time_between_locations(
            coordinates[point_id],
            coordinates[route[ix+1]]
        )

    return score

In [11]:
route = create_guess(
    list(supply_locations.keys())
)

print(route)


['Clinic_A', 'Clinic_I', 'Central_Hospital', 'Clinic_H', 'Clinic_F', 'Clinic_B', 'Clinic_E', 'Clinic_K', 'Clinic_J', 'Clinic_D', 'Clinic_C', 'Clinic_G', 'Clinic_A']


In [12]:
fitness_score(route)

np.float32(28.942675)

In [13]:
# Test the fitness function
route = create_guess(
    list(supply_locations.keys())
)
print(route)
print(
    "Total Route Time:",
    fitness_score(route)
)

['Clinic_E', 'Clinic_D', 'Clinic_C', 'Clinic_F', 'Clinic_B', 'Clinic_G', 'Clinic_I', 'Clinic_A', 'Clinic_K', 'Clinic_H', 'Clinic_J', 'Central_Hospital', 'Clinic_E']
Total Route Time: 31.820211


In [14]:
# Create the initial population
test_generation = create_generation(
    list(supply_locations.keys()),
    population=10
)
print(test_generation)

[['Clinic_B', 'Clinic_E', 'Clinic_D', 'Central_Hospital', 'Clinic_I', 'Clinic_G', 'Clinic_A', 'Clinic_F', 'Clinic_H', 'Clinic_J', 'Clinic_C', 'Clinic_K', 'Clinic_B'], ['Clinic_H', 'Clinic_E', 'Clinic_F', 'Clinic_J', 'Clinic_D', 'Clinic_B', 'Clinic_I', 'Clinic_C', 'Central_Hospital', 'Clinic_K', 'Clinic_G', 'Clinic_A', 'Clinic_H'], ['Clinic_A', 'Central_Hospital', 'Clinic_C', 'Clinic_F', 'Clinic_B', 'Clinic_E', 'Clinic_G', 'Clinic_K', 'Clinic_I', 'Clinic_J', 'Clinic_H', 'Clinic_D', 'Clinic_A'], ['Clinic_G', 'Clinic_J', 'Clinic_I', 'Clinic_F', 'Clinic_A', 'Clinic_E', 'Clinic_K', 'Clinic_H', 'Central_Hospital', 'Clinic_C', 'Clinic_B', 'Clinic_D', 'Clinic_G'], ['Clinic_C', 'Clinic_I', 'Clinic_A', 'Clinic_J', 'Clinic_B', 'Central_Hospital', 'Clinic_E', 'Clinic_H', 'Clinic_F', 'Clinic_D', 'Clinic_G', 'Clinic_K', 'Clinic_C'], ['Clinic_E', 'Clinic_D', 'Clinic_C', 'Clinic_H', 'Clinic_I', 'Central_Hospital', 'Clinic_A', 'Clinic_F', 'Clinic_G', 'Clinic_J', 'Clinic_B', 'Clinic_K', 'Clinic_E'], ['C

In [15]:
# Test check fitness
def check_fitness(routes):
    fitness_indicator = []
    for route in routes:
        fitness_indicator.append(
            (
            route,
            fitness_score(route)
            )
        )
    return fitness_indicator

results = check_fitness(test_generation)
results[:3]

[(['Clinic_B',
   'Clinic_E',
   'Clinic_D',
   'Central_Hospital',
   'Clinic_I',
   'Clinic_G',
   'Clinic_A',
   'Clinic_F',
   'Clinic_H',
   'Clinic_J',
   'Clinic_C',
   'Clinic_K',
   'Clinic_B'],
  np.float32(32.136776)),
 (['Clinic_H',
   'Clinic_E',
   'Clinic_F',
   'Clinic_J',
   'Clinic_D',
   'Clinic_B',
   'Clinic_I',
   'Clinic_C',
   'Central_Hospital',
   'Clinic_K',
   'Clinic_G',
   'Clinic_A',
   'Clinic_H'],
  np.float32(28.99683)),
 (['Clinic_A',
   'Central_Hospital',
   'Clinic_C',
   'Clinic_F',
   'Clinic_B',
   'Clinic_E',
   'Clinic_G',
   'Clinic_K',
   'Clinic_I',
   'Clinic_J',
   'Clinic_H',
   'Clinic_D',
   'Clinic_A'],
  np.float32(30.46141))]

In [16]:
def get_breeders_from_generation(
    guesses,
    take_best_N=5,
    take_random_N=2
):

    fit_scores = check_fitness(guesses)

    sorted_guesses = sorted(
        fit_scores,
        key=lambda x: x[1]
    )

    new_generation = [
        x[0]
        for x in sorted_guesses[:take_best_N]
    ]

    best_guess = new_generation[0]

    for _ in range(take_random_N):
        ix = np.random.randint(len(guesses))
        new_generation.append(
            guesses[ix]
        )

    np.random.shuffle(new_generation)

    return new_generation, best_guess
breeders, best_route = get_breeders_from_generation(
    test_generation
)

print(best_route)


['Clinic_H', 'Clinic_E', 'Clinic_F', 'Clinic_J', 'Clinic_D', 'Clinic_B', 'Clinic_I', 'Clinic_C', 'Central_Hospital', 'Clinic_K', 'Clinic_G', 'Clinic_A', 'Clinic_H']


In [17]:
# CrossOver

def make_child(parent1, parent2):
    child = [-99] * len(parent1)

    half = len(parent1) // 2

    for i in range(half):
        child[i] = parent1[i]
    for i in range(len(child)):
        if child[i] == -99:
            for gene in parent2:
                if gene not in child:
                    child[i] = gene
                    break
    child[-1] = child[0]
    return child

# test
child = make_child(
    breeders[0],
    breeders[1]
)
print(child)

['Clinic_C', 'Clinic_I', 'Clinic_A', 'Clinic_J', 'Clinic_B', 'Central_Hospital', 'Clinic_H', 'Clinic_G', 'Clinic_D', 'Clinic_F', 'Clinic_E', 'Clinic_K', 'Clinic_C']


In [18]:
def make_children(
    old_generation,
    children_per_couple=2
):

    midpoint = len(old_generation)//2

    next_generation = []

    for ix, parent in enumerate(
        old_generation[:midpoint]
    ):

        for _ in range(children_per_couple):

            next_generation.append(
                make_child(
                    parent,
                    old_generation[-ix-1]
                )
            )

    return next_generation

In [19]:
# Test
children = make_children(
    breeders
)
print(len(children))

6


In [20]:
# Evolution Loop
current_generation = create_generation(
    list(supply_locations.keys()),
    population=300
)

best_score_ever = float("inf")
best_route_ever = None

for generation in range(100):

    breeders, best_route = (
        get_breeders_from_generation(
            current_generation,
            take_best_N=30,
            take_random_N=10
        )
    )

    current_score = fitness_score(best_route)

    if current_score < best_score_ever:
        best_score_ever = current_score
        best_route_ever = best_route

    current_generation = make_children(
        breeders,
        children_per_couple=2
    )

    if generation % 10 == 0:

        print(f"Generation {generation}")

        print("Current Best Route:")
        print(best_route)

        print("Current Score:")
        print(current_score)

print("\n===== FINAL RESULT =====")

print("Best Route Ever:")
print(best_route_ever)

print("Best Score Ever:")
print(best_score_ever)

Generation 0
Current Best Route:
['Clinic_F', 'Clinic_H', 'Clinic_E', 'Clinic_A', 'Central_Hospital', 'Clinic_G', 'Clinic_K', 'Clinic_D', 'Clinic_C', 'Clinic_I', 'Clinic_B', 'Clinic_J', 'Clinic_F']
Current Score:
26.538397
Generation 10
Current Best Route:
['Clinic_G', 'Clinic_A', 'Clinic_C', 'Clinic_K', 'Clinic_B', 'Clinic_J', 'Clinic_I', 'Clinic_D', 'Clinic_F', 'Clinic_E', 'Central_Hospital', 'Clinic_H', 'Clinic_G']
Current Score:
27.266083
Generation 20
Current Best Route:
['Clinic_K', 'Clinic_B', 'Clinic_J', 'Clinic_G', 'Clinic_F', 'Clinic_A', 'Clinic_I', 'Clinic_C', 'Clinic_D', 'Clinic_E', 'Central_Hospital', 'Clinic_H', 'Clinic_K']
Current Score:
27.355053
Generation 30
Current Best Route:
['Clinic_K', 'Clinic_B', 'Clinic_J', 'Clinic_G', 'Clinic_F', 'Clinic_A', 'Clinic_I', 'Clinic_C', 'Clinic_D', 'Clinic_H', 'Central_Hospital', 'Clinic_E', 'Clinic_K']
Current Score:
27.679522
Generation 40
Current Best Route:
['Clinic_K', 'Clinic_B', 'Clinic_J', 'Clinic_G', 'Clinic_F', 'Clinic_A'

In [21]:
# Final result section
print("\n" + "="*50)
print("EMERGENCY MEDICAL SUPPLY OPTIMIZATION")
print("="*50)

print("\nBest Route:")

for stop in best_route_ever:
    print("→", stop)

print("\nPredicted Total Travel Time:")
print(f"{best_score_ever:.2f} minutes")


EMERGENCY MEDICAL SUPPLY OPTIMIZATION

Best Route:
→ Clinic_F
→ Clinic_H
→ Clinic_E
→ Clinic_A
→ Central_Hospital
→ Clinic_G
→ Clinic_K
→ Clinic_D
→ Clinic_C
→ Clinic_I
→ Clinic_B
→ Clinic_J
→ Clinic_F

Predicted Total Travel Time:
26.54 minutes


In [22]:
# Create the dataframe
import pandas as pd

route_df = pd.DataFrame({
    "Visit Order": range(1, len(best_route_ever)+1),
    "Location": best_route_ever
})

route_df

,Visit Order,Location
0,1,Clinic_F
1,2,Clinic_H
2,3,Clinic_E
3,4,Clinic_A
4,5,Central_Hospital
5,6,Clinic_G
6,7,Clinic_K
7,8,Clinic_D
8,9,Clinic_C
9,10,Clinic_I


In [23]:
# Save the result
route_df.to_csv(
    "optimized_supply_route.csv",
    index=False
)

print("Route saved successfully")

Route saved successfully
